# 최종 선정 모델 재현 노트북
독립 실행 가능 · 외부 노트북 import 없음

| 종목 | 최종 모델 | 비고 |
|------|----------|------|
| 삼성전자·기아·한화에어로·LIG넥스원·KB금융·신한지주·SK이노베이션 | pkl 로드 | `saved_models/final/` |
| SK하이닉스·현대차·LG화학 | PatchTST | `patchtst_v18_model.pkl` |

**사전 준비**
- `.env` — KRX_ID, KRX_PW 설정
- `data/model_v2/*.parquet` — 종목별 가격 데이터
- `data/saved_models/final/*.pkl` — 우리 모델 7종목
- `model/patchtst_v18_model.pkl` — PatchTST state_dict


In [ ]:
import os, warnings, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from scipy.stats import spearmanr
import torch
import torch.nn as nn
import FinanceDataReader as fdr
from pykrx import stock as pykrx_stock
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression

# .env 위치: inference_package/.env
# KRX_ID, KRX_PW 설정 필요 (pykrx KOSPI 데이터용)
load_dotenv(Path('../.env'), override=True)
warnings.filterwarnings('ignore')

# ── 경로 설정 ─────────────────────────────────────────────────
ROOT      = Path('..')
DATA_DIR  = ROOT / 'data' / 'model_v2'
SAVE_DIR  = ROOT / 'data' / 'saved_models'
PTST_PKL  = ROOT / 'model' / 'patchtst_v18_model.pkl'

# ── 상수 ──────────────────────────────────────────────────────
TODAY      = pd.Timestamp('2026-06-02')   # ← 실행 시 날짜로 변경
EVAL_START = TODAY - pd.DateOffset(months=1)
HORIZON    = 5
BASE_FEAT  = ['ret_1d','ret_5d','ret_20d','vol_norm',
              'kospi_ret','sp500_ret','ndx_ret','usdkrw_ret','vix_chg']
ALL_FEAT   = BASE_FEAT + ['regime_prob','regime_duration','regime_change']

# ── 종목 정의 + 최종 모델 매핑 ────────────────────────────────
TICKERS = {
    '005930': '삼성전자',
    '000660': 'SK하이닉스',
    '005380': '현대차',
    '000270': '기아',
    '012450': '한화에어로',
    '079550': 'LIG넥스원',
    '105560': 'KB금융',
    '055550': '신한지주',
    '051910': 'LG화학',
    '096770': 'SK이노베이션',
}
# 최종 채택 모델 타입
FINAL_TYPE = {
    '005930': 'ours',    '000660': 'patchtst', '005380': 'patchtst',
    '000270': 'ours',    '012450': 'ours',      '079550': 'ours',
    '105560': 'ours',    '055550': 'ours',      '051910': 'patchtst',
    '096770': 'ours',
}
# PatchTST pkl 내 키 이름
PTST_KEY = {
    '000660': 'SK Hynix',      '005380': 'Hyundai Motor',
    '051910': 'LG Chem',
}

print('설정 완료')
print(f'평가 기간: {EVAL_START.date()} ~ {TODAY.date()}')


## PatchTST 모델 클래스 (팀원 구현, AdvancedPatchTSTModel)

In [ ]:
class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(1, 1, num_features))
        self.bias   = nn.Parameter(torch.zeros(1, 1, num_features))
    def forward(self, x, mode):
        if mode == 'norm':
            self.mean = x.mean(1, keepdim=True)
            self.std  = x.std(1, keepdim=True) + self.eps
            return (x - self.mean) / self.std * self.weight + self.bias
        return (x - self.bias) / self.weight * self.std + self.mean


class AdvancedPatchTSTModel(nn.Module):
    """
    입력: (batch, seq_len=512, c_in=9)  — 512 거래일 × 9개 피처
    출력: D+1~D+5 로그 수익률 5개 → D+5 합산 사용
    파라미터: 125,271개 (종목별 독립 학습)
    """
    def __init__(self, c_in=9, seq_len=512, pred_len=5,
                 patch_len=16, stride=8, d_model=64,
                 n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.c_in      = c_in
        self.patch_len = patch_len
        self.stride    = stride
        num_patches    = (seq_len - patch_len) // stride + 1

        self.revin           = RevIN(c_in)
        self.patch_embedding = nn.Linear(patch_len, d_model)
        self.pos_embedding   = nn.Parameter(torch.zeros(1, num_patches, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, activation='gelu')
        self.transformer_encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x):
        B = x.shape[0]
        x  = self.revin(x, 'norm')
        xc = x.permute(0,2,1).reshape(B * self.c_in, x.shape[1])
        patches = xc.unfold(1, self.patch_len, self.stride)
        emb = self.patch_embedding(patches) + self.pos_embedding
        enc = self.transformer_encoder(emb)
        out = self.head(enc.reshape(B * self.c_in, -1)).reshape(B, self.c_in, -1)
        return out[:, 0, :].sum(dim=-1)   # D+5 합산 log return


print('PatchTST 클래스 정의 완료')


## 모델 로드

In [ ]:
MODELS = {}   # ticker → 모델 객체 (ours: sklearn/lgbm, patchtst: nn.Module)

# 우리 모델 (pkl)
for ticker, name in TICKERS.items():
    if FINAL_TYPE[ticker] != 'ours': continue
    pkl_path = SAVE_DIR / f'{ticker}.pkl'
    if not pkl_path.exists():
        print(f'{name}: pkl 없음 — {pkl_path}'); continue
    with open(pkl_path, 'rb') as f:
        MODELS[ticker] = pickle.load(f)
    n = getattr(MODELS[ticker], 'n_features_in_', '?')
    print(f'{name}: {type(MODELS[ticker]).__name__}  (피처={n}개)  ✓')

# PatchTST (state_dict)
with open(PTST_PKL, 'rb') as f:
    ptst_state_dicts = pickle.load(f)

for ticker, key in PTST_KEY.items():
    name = TICKERS[ticker]
    if key not in ptst_state_dicts:
        print(f'{name}: state_dict 키 없음 ({key})'); continue
    m = AdvancedPatchTSTModel()
    m.load_state_dict(ptst_state_dicts[key], strict=False)
    m.eval()
    MODELS[ticker] = m
    print(f'{name}: PatchTST  (125,271 params)  ✓')

print(f'\n로드 완료: {len(MODELS)}/10종목')


## 데이터 준비

In [ ]:
# 매크로 데이터
kospi_raw = pykrx_stock.get_index_ohlcv_by_date(
    '20210101', TODAY.strftime('%Y%m%d'), '1001', name_display=False)
kospi_raw.index = pd.to_datetime(kospi_raw.index)
kospi_ret = np.log(kospi_raw['종가'] / kospi_raw['종가'].shift(1)).rename('kospi_ret')

macro = pd.DataFrame()
for sym, col in [('S&P500','sp500'),('VIX','vix'),('NDX','ndx'),('USD/KRW','usdkrw')]:
    d = fdr.DataReader(sym, '20210101')[['Close']].rename(columns={'Close': col})
    d.index = pd.to_datetime(d.index)
    macro = d if macro.empty else macro.join(d, how='outer')
macro = macro.ffill()
for c, r in [('sp500','sp500_ret'),('ndx','ndx_ret'),('usdkrw','usdkrw_ret')]:
    macro[r] = np.log(macro[c] / macro[c].shift(1))
macro['vix_chg'] = macro['vix'].diff()
macro = macro[['sp500_ret','ndx_ret','usdkrw_ret','vix_chg']].replace([np.inf,-np.inf], np.nan)

# 종목별 데이터 + MSAR regime 피처 (전 종목 계산)
STOCK_DATA = {}
for ticker, name in TICKERS.items():
    try:
        df = pd.read_parquet(DATA_DIR / f'{ticker}.parquet')
        df.index = pd.to_datetime(df.index); df = df.sort_index()
        df['ret_1d']   = np.log(df['close'] / df['close'].shift(1))
        df['ret_5d']   = np.log(df['close'] / df['close'].shift(5))
        df['ret_20d']  = np.log(df['close'] / df['close'].shift(20))
        df['vol_norm'] = df['volume'] / df['volume'].rolling(20).mean()
        df['target']   = np.log(df['close'].shift(-HORIZON) / df['close'])
        df = df.join(kospi_ret, how='left').join(macro, how='left') \
               .replace([np.inf,-np.inf], np.nan)

        # MSAR regime 피처
        ret_s = df['ret_1d'].dropna()
        try:
            res = MarkovAutoregression(
                ret_s.values, k_regimes=2, order=1,
                switching_ar=False, switching_variance=True
            ).fit(disp=False, maxiter=150)
            fp   = res.filtered_marginal_probabilities
            avgs = [float(np.average(ret_s.values[-len(fp):], weights=fp[:,k])) for k in range(2)]
            bull = int(np.argmax(avgs))
            prob = pd.Series(fp[:,bull], index=ret_s.index[-len(fp):]).reindex(df.index).ffill()
            df['regime_prob']     = prob
            df['regime_duration'] = (
                (prob>0.5).astype(int)
                .groupby((prob>0.5).ne((prob>0.5).shift()).cumsum())
                .cumcount() + 1
            )
            df['regime_change'] = (prob>0.5).ne((prob>0.5).shift()).astype(int)
        except:
            df['regime_prob'] = 0.5; df['regime_duration'] = 1; df['regime_change'] = 0

        STOCK_DATA[ticker] = df
        m = MODELS.get(ticker)
        n = getattr(m, 'n_features_in_', 9) if m and not isinstance(m, AdvancedPatchTSTModel) else 9
        print(f'{name}: {len(df)}일  모델피처={n}개')
    except Exception as e:
        print(f'{name}: 실패 {e}')

print('\n데이터 준비 완료')


## 최근 1달 평가 — MAPE + Rank IC

In [ ]:
SEQ_LEN = 512   # PatchTST 시퀀스 길이

RESULTS = {}

for ticker, name in TICKERS.items():
    df  = STOCK_DATA.get(ticker)
    m   = MODELS.get(ticker)
    if df is None or m is None:
        print(f'{name}: 데이터/모델 없음'); continue

    is_ptst = isinstance(m, AdvancedPatchTSTModel)
    close   = df['close']
    ret1d   = df['ret_1d'].dropna()

    # 피처 컬럼 결정
    if not is_ptst:
        n = getattr(m, 'n_features_in_', 9)
        feat_cols = [f for f in ALL_FEAT if f in df.columns][:n]
    else:
        feat_cols = BASE_FEAT

    # 평가 날짜: 최근 1달, D+5 실제 가격 존재
    eval_dates = [
        d for d in df[(df.index >= EVAL_START) & (df.index < TODAY)].index
        if (d + pd.offsets.BDay(5)) <= close.index[-1]
    ]
    if not eval_dates:
        print(f'{name}: 평가 날짜 없음'); continue

    pred_rs, actual_rs, pred_pxs, actual_pxs = [], [], [], []

    for base_dt in eval_dates:
        tgt_dt    = base_dt + pd.offsets.BDay(5)
        base_px   = float(close.asof(base_dt))
        actual_px = float(close.asof(tgt_dt))
        if np.isnan(base_px) or np.isnan(actual_px): continue
        actual_r  = np.log(actual_px / base_px)

        try:
            if not is_ptst:
                row = df[df.index <= base_dt].dropna(subset=feat_cols).tail(1)
                if row.empty: continue
                X  = np.nan_to_num(row[feat_cols].values)
                pr = float(m.predict(X)[0])
            else:
                seq = df[df.index <= base_dt].dropna(subset=BASE_FEAT).tail(SEQ_LEN)
                if len(seq) < SEQ_LEN: continue
                sv  = seq[BASE_FEAT].values
                xt  = torch.tensor(sv, dtype=torch.float32).unsqueeze(0)
                with torch.no_grad(): raw = float(m(xt).item())
                mu  = float(sv[:,0].mean()); sig = float(sv[:,0].std()) + 1e-8
                pr  = float(np.clip(raw * sig + mu, -0.15, 0.15))
        except: continue

        pred_px = base_px * np.exp(np.clip(pr, -0.3, 0.3))
        pred_rs.append(pr); actual_rs.append(actual_r)
        pred_pxs.append(pred_px); actual_pxs.append(actual_px)

    if len(pred_rs) < 3:
        print(f'{name}: 유효 샘플 부족 (n={len(pred_rs)})'); continue

    pred_rs    = np.array(pred_rs)
    actual_rs  = np.array(actual_rs)
    actual_pxs = np.array(actual_pxs)
    pred_pxs   = np.array(pred_pxs)

    mape = float(np.mean(np.abs(pred_pxs - actual_pxs) / actual_pxs) * 100)
    ic, pval = spearmanr(pred_rs, actual_rs)

    model_tag = 'PatchTST' if is_ptst else type(m).__name__
    RESULTS[ticker] = {
        'name': name, 'model': model_tag, 'n': len(pred_rs),
        'mape': mape, 'ic': float(ic), 'pval': float(pval),
    }

# ── 결과 출력 ────────────────────────────────────────────────
print(f'\n평가 기간: {EVAL_START.date()} ~ {TODAY.date()}')
print('=' * 72)
print(f'{"종목":<12} {"모델":<26} {"n":>4}  {"MAPE":>7}  {"Rank IC":>9}  {"p-value":>8}')
print('=' * 72)
for ticker in TICKERS:
    if ticker not in RESULTS: continue
    r = RESULTS[ticker]
    sig = '**' if r['pval'] < 0.05 else '  '
    print(f'{r["name"]:<12} {r["model"]:<26} {r["n"]:>4}  '
          f'{r["mape"]:>6.2f}%  {r["ic"]:>+9.4f}  {r["pval"]:>8.4f} {sig}')
print('=' * 72)
print('** p < 0.05')

# CSV 저장
out = pd.DataFrame(RESULTS).T[['name','model','n','mape','ic','pval']]
out.to_csv(ROOT / 'data' / 'inference_results.csv', encoding='utf-8-sig')
print('\n저장: data/inference_results.csv')


In [ ]:
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.stats import norm as sp_norm

TODAY        = pd.Timestamp('2026-06-02')
HIST_START   = TODAY - pd.DateOffset(months=3)
VAL_START    = TODAY - pd.DateOffset(weeks=6)    # 검증 구간 2주
CI_PCT       = 0.8
CI_Z         = sp_norm.ppf((1 + CI_PCT) / 2)
VOL_DAYS     = 20
ALL_FEAT_SVC = BASE_FEAT + ['regime_prob', 'regime_duration', 'regime_change']

C_PRICE   = '#1a1a2e'
C_DOT_ACT = '#2d6a4f'
C_BG      = '#f8f9fa'
C_PRED    = '#4361ee'
C_PATCH   = '#e63946'

fig = plt.figure(figsize=(20, 30), facecolor='white')
n_cols, n_rows = 2, (len(TICKERS) + 1) // 2
gs = GridSpec(n_rows, n_cols, figure=fig, hspace=0.48, wspace=0.12)

for idx, ticker in enumerate(TICKERS.keys()):
    name   = TICKERS[ticker]
    res    = RESULTS.get(ticker)
    df     = STOCK_DATA.get(ticker)
    our_m  = MODELS.get(ticker)
    ptst_m = PATCHTST_MODELS.get(ticker)
    if df is None or res is None: continue

    is_ptst = (res['winner'] == 'PatchTST')
    accent  = C_PATCH if is_ptst else C_PRED
    win_mape = res['ptst_mape'] if is_ptst else res['our_mape']

    ax = fig.add_subplot(gs[idx // n_cols, idx % n_cols])
    ax.set_facecolor(C_BG)
    for sp in ax.spines.values(): sp.set_edgecolor('#dee2e6')

    close = df['close']
    ret1d = df['ret_1d'].dropna()
    n = getattr(our_m, 'n_features_in_', 9)
    feat_cols = [f for f in ALL_FEAT_SVC if f in df.columns][:n]

    def get_pred_px(bd, bpx):
        row = df[df.index <= bd].dropna(subset=feat_cols).tail(1)
        if row.empty: return None, None
        X = np.nan_to_num(row[feat_cols].values)
        try:
            if not is_ptst:
                pr = float(our_m.predict(X)[0])
            else:
                seq = df[df.index <= bd].dropna(subset=BASE_FEAT).tail(SEQ_LEN)
                if len(seq) < SEQ_LEN: return None, None
                sv = seq[BASE_FEAT].values
                xt = torch.tensor(sv, dtype=torch.float32).unsqueeze(0)
                with torch.no_grad(): raw = float(ptst_m(xt).item())
                mu = float(sv[:,0].mean()); sig = float(sv[:,0].std()) + 1e-8
                pr = np.clip(raw * sig + mu, -0.15, 0.15)
            rec = ret1d[ret1d.index <= bd].tail(VOL_DAYS)
            sigma = float(rec.std()) if len(rec) > 3 else 0.015
            ci_h  = CI_Z * sigma * np.sqrt(HORIZON)
            ppx   = bpx * np.exp(np.clip(pr, -0.3, 0.3))
            return ppx, ci_h
        except: return None, None

    # ── 1. 3개월 실주가 ────────────────────────────────────
    hist = close[(close.index >= HIST_START) & (close.index <= TODAY)]
    ax.plot(hist.index, hist.values, color=C_PRICE, lw=2.0, zorder=4)

    # ── 2. 검증 구간 2주: 예측 + CI ───────────────────────
    val_dates = [
        d for d in df[(df.index >= VAL_START) & (df.index < TODAY)].index
        if (d + pd.offsets.BDay(5)) <= close.index[-1]
    ]
    if val_dates:
        anc_dt = val_dates[0]
        anc_px = float(close.asof(anc_dt))
        v_dts  = [anc_dt]; v_pxs = [anc_px]; v_lo = [anc_px]; v_hi = [anc_px]
        for bd in val_dates:
            td  = bd + pd.offsets.BDay(5)
            bpx = float(close.asof(bd))
            apx = float(close.asof(td))
            if np.isnan(bpx) or np.isnan(apx): continue
            ppx, ci_h = get_pred_px(bd, bpx)
            if ppx is None: continue
            ax.scatter([td], [apx], color=C_DOT_ACT, s=28, zorder=6, alpha=0.85)
            v_dts.append(td); v_pxs.append(ppx)
            v_lo.append(bpx * np.exp(np.log(ppx / bpx) - ci_h))
            v_hi.append(bpx * np.exp(np.log(ppx / bpx) + ci_h))
        if len(v_dts) > 1:
            ax.fill_between(v_dts, v_lo, v_hi, color=accent, alpha=0.10, zorder=2)
            ax.plot(v_dts, v_pxs, color=accent, lw=1.4, ls='--', alpha=0.7, zorder=3)
            ax.scatter(v_dts[1:], v_pxs[1:], color=accent, s=22,
                       marker='^', zorder=5, alpha=0.8)

    # ── 3. 미래 D+5 예측 + CI ─────────────────────────────
    today_px = float(close.asof(TODAY))
    today_dt = close.index[close.index <= TODAY][-1]
    fut_dts  = [today_dt]; fut_pxs = [today_px]
    f_lo     = [today_px]; f_hi    = [today_px]
    for h in range(1, 6):
        bd  = TODAY - pd.offsets.BDay(5 - h)
        td  = TODAY + pd.offsets.BDay(h)
        bpx = float(close.asof(bd))
        if np.isnan(bpx): continue
        ppx, ci_h = get_pred_px(bd, bpx)
        if ppx is None: continue
        fut_dts.append(td); fut_pxs.append(ppx)
        f_lo.append(bpx * np.exp(np.log(ppx / bpx) - ci_h))
        f_hi.append(bpx * np.exp(np.log(ppx / bpx) + ci_h))
    if len(fut_dts) > 1:
        ax.fill_between(fut_dts, f_lo, f_hi, color=accent, alpha=0.14, zorder=3)
        ax.plot(fut_dts, fut_pxs, color=accent, lw=2.2, ls='--', zorder=5)
        for h_i, (dt, px) in enumerate(zip(fut_dts[1:], fut_pxs[1:]), 1):
            ax.scatter([dt], [px], color=accent, s=60, zorder=7,
                       edgecolors='white', lw=1.3)
            ax.annotate(f'D+{h_i}', xy=(dt, px), xytext=(0, 8),
                        textcoords='offset points', ha='center',
                        fontsize=7.5, color=accent, fontweight='bold')

    # ── 구분선 + 가격 annotation ──────────────────────────
    ax.axvline(TODAY, color='#6c757d', lw=1.2, ls=':', alpha=0.8, zorder=6)
    ax.annotate(f'  {today_px:,.0f}원', xy=(TODAY, today_px),
                fontsize=8.5, color=C_PRICE, va='center', fontweight='bold')
    if len(fut_pxs) > 1:
        d5  = fut_pxs[-1]
        pct = (d5 - today_px) / today_px * 100
        col = '#e63946' if pct < 0 else '#2d6a4f'
        ax.annotate(f'D+5  {d5:,.0f}원  ({pct:+.1f}%)',
                    xy=(fut_dts[-1], d5), xytext=(-4, -15),
                    textcoords='offset points', ha='right',
                    fontsize=8, color=col, fontweight='bold')

    # ── 제목: 최종 선정 모델 + 해당 모델 MAPE만 표시 ────
    mt = 'PatchTST' if is_ptst else type(our_m).__name__
    ax.set_title(
        f'{name}  ·  {mt}  ·  MAPE {win_mape:.1f}%',
        fontsize=10.5, fontweight='bold', color=accent, pad=8)

    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=0))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, ha='center', fontsize=8.5)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, which='major', alpha=0.15, color='#adb5bd')
    ax.grid(True, which='minor', alpha=0.05, color='#adb5bd')

    handles = [
        mpatches.Patch(color=C_PRICE,   label='실주가'),
        mpatches.Patch(color=C_DOT_ACT, label='검증 실제값'),
        mpatches.Patch(color=accent,    label=f'예측 ({mt})'),
        mpatches.Patch(color=accent, alpha=0.2, label=f'{int(CI_PCT*100)}% CI'),
    ]
    ax.legend(handles=handles, fontsize=7.5, loc='upper left',
              framealpha=0.88, edgecolor='#dee2e6', ncol=2)

our_n  = sum(1 for r in RESULTS.values() if r['winner'] == 'OUR')
ptst_n = sum(1 for r in RESULTS.values() if r['winner'] == 'PatchTST')
fig.suptitle(
    f'WHAI  |  D+5 주가 예측  (기준일: {TODAY.date()})  '
    f'·  우리 모델 {our_n}종목  /  PatchTST {ptst_n}종목',
    fontsize=14, fontweight='bold', color='#1a1a2e', y=1.003)

plt.savefig('../data/service_viz_2w.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('저장: data/service_viz_2w.png')